# Session 8 notebook: does it survive trading costs?

**Question.** Every Sharpe ratio so far in this course was before trading costs. A strategy that trades a lot can look great on paper and lose money in practice. How much does each strategy trade, how large can costs get before its profit is gone, and does a popular overlay, volatility targeting, survive the test?

| Part | What you do | When |
|---|---|---|
| 1 | How much do six anomaly strategies trade? | before class, 10 min |
| 2 | Volatility-managed factors, as in Moreira and Muir (2017) | in class |
| 3 | The same strategy in real time (Cederburg et al., 2020) | in class |
| 4 | Break-even costs | in class |
| 5 | Discuss | in class |

**Data.** Part 1 uses `week3_data.zip` from Week 3 (the link is on the course notes page). Parts 2&ndash;4 download Ken French's daily and monthly factors directly.

Cells marked **Your turn** work as written. Change the value, rerun, and say what changed and why.

## Part 0. Setup (before class)

In [ ]:
import io, re, zipfile, urllib.request
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from pathlib import Path
pd.set_option("display.width", 160, "display.max_columns", 30, "display.float_format", "{:,.2f}".format)

KF = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/"

def kf(name, table=0):
    """Read one table from a Ken French CSV zip. Returns decimals, indexed by month ('YYYY-MM') or by day."""
    z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(KF + name + "_CSV.zip").read()))
    lines = z.read(z.namelist()[0]).decode("latin1").splitlines()
    i = [k for k, l in enumerate(lines) if l.startswith(",")][table]
    rows = []
    for l in lines[i + 1:]:
        if not l.strip() or not l.strip()[0].isdigit():
            break
        rows.append(l)
    df = pd.read_csv(io.StringIO("\n".join([lines[i]] + rows)), index_col=0)
    df.columns = [c.strip() for c in df.columns]
    idx = df.index.astype(str).str.strip()
    df.index = pd.to_datetime(idx, format="%Y%m%d") if len(idx[0]) == 8 else (idx.str[:4] + "-" + idx.str[4:])
    return df.replace([-99.99, -999], np.nan) / 100

def sharpe(r): return r.mean() / r.std() * np.sqrt(12)

# The class data (week3_data.zip). The link is on the course notes page, not in this public notebook.
DATA_ZIP_LINK = ""     # <- paste the Google Drive link to week3_data.zip between the quotes, then run this cell

DATA_DIR = Path("data")
if not (DATA_DIR / "w3_panel.parquet").exists():
    fid = re.search(r"/d/([\w-]+)|[?&]id=([\w-]+)", DATA_ZIP_LINK)
    assert fid, "Paste the week3_data.zip link from the course notes page into DATA_ZIP_LINK above, then rerun this cell."
    fid = fid.group(1) or fid.group(2)
    urllib.request.urlretrieve(f"https://drive.usercontent.google.com/download?id={fid}&export=download&confirm=t", "week3_data.zip")
    assert zipfile.is_zipfile("week3_data.zip"), "The download did not return a zip. Check that the link is shared with you, and try again."
    zipfile.ZipFile("week3_data.zip").extractall(".")
panel = pd.read_parquet(DATA_DIR / "w3_panel.parquet")
print(f"panel: {len(panel):,} stock-months, {panel.month.min()} to {panel.month.max()}")

## Part 1. How much does each strategy trade? (before class)

Each strategy is the Week 2 recipe: each month, sort stocks on a signal, buy the top fifth and short the bottom fifth, equal-weighted. That is $1 long and $1 short. When the sort changes, you trade.

- **Dollars traded per month**: the sum of all weight changes, buys plus sells, on both legs.
- **One-sided turnover**: the share of each leg replaced in a month (dollars traded / 4). Novy-Marx and Velikov's rule of thumb: most anomalies with one-sided turnover below 50% a month survive costs; few above it do.
- **Break-even cost**: the average monthly return divided by dollars traded per month. If each dollar you trade costs more than this, the strategy loses money.

The signals are Chen&ndash;Zimmermann's, signed so that a higher value means a higher expected return. `RealizedVol` is volatility over the past month, signed so the top fifth holds the *least* volatile stocks.

In [ ]:
STRATEGIES = {"BMdec": "value (book-to-market)", "Mom12m": "momentum (12 months)", "RealizedVol": "low volatility (1 month)",
              "MRreversal": "medium-run reversal", "AssetGrowth": "low asset growth", "Illiquidity": "illiquidity"}
panel["top1000"] = panel.groupby("month")["me_prev"].rank(ascending=False) <= 1000

def long_short(df, signal):
    q = df.groupby("month")[signal].transform(lambda x: pd.qcut(x.rank(method="first"), 5, labels=False))
    n = df.assign(q=q).groupby(["month", "q"])["permno"].transform("count")
    w = np.where(q == 4, 1 / n, np.where(q == 0, -1 / n, 0.0))
    W = pd.Series(w, index=pd.MultiIndex.from_arrays([df.month, df.permno])).unstack(fill_value=0.0)
    ret = pd.Series(w * df.y.values).groupby(df.month.values).sum()
    traded = W.diff().abs().sum(axis=1).iloc[1:].mean()
    return {"return, % a year": ret.mean() * 1200, "Sharpe": sharpe(ret), "dollars traded a month": traded,
            "one-sided turnover": traded / 4, "break-even cost, bp": ret.mean() / traded * 1e4}

UNIVERSE = "all"      # Your turn: "top1000" keeps only the 1,000 largest stocks each month
df = panel if UNIVERSE == "all" else panel[panel.top1000]
turnover = pd.DataFrame({label: long_short(df, s) for s, label in STRATEGIES.items()}).T
turnover

**Check yourself.**
1. Which strategy trades least, and why does a book-to-market sort change slowly?
2. Low volatility here uses one month of daily returns. How would the turnover change with a five-year window, as many low-volatility funds use?
3. Rerun with the 1,000 largest stocks. Which strategies lose most of their return? Small stocks are also the expensive ones to trade.

## Part 2. Volatility-managed factors (in class)

Moreira and Muir (2017) scale a factor's position each month by the inverse of last month's realized variance:

> f<sup>&sigma;</sup><sub>t</sub> = (c / &sigma;&#770;&sup2;<sub>t&minus;1</sub>) &times; f<sub>t</sub>

where &sigma;&#770;&sup2;<sub>t&minus;1</sub> is the sum of squared *daily* factor returns last month. When markets are calm you hold more; when they are turbulent you hold less. The constant c makes the managed factor as volatile as the original over the full sample.

The test is a regression of the managed factor on the original. A positive intercept (alpha) means the managed version adds something an investor in the original factor doesn't have.

In [ ]:
daily = kf("F-F_Research_Data_Factors_daily").join(kf("F-F_Momentum_Factor_daily"), how="inner")
monthly = kf("F-F_Research_Data_Factors").join(kf("F-F_Momentum_Factor"), how="inner")
daily["month"] = daily.index.strftime("%Y-%m")
FACTORS = ["Mkt-RF", "HML", "Mom"]
rv = daily.groupby("month")[FACTORS].apply(lambda x: (x**2).sum())       # realized variance in each month

def managed(f, start="1927-01"):
    r = monthly[f].loc[start:]
    inv_var = (1 / rv[f]).shift(1).reindex(r.index)                      # last month's realized variance
    return pd.concat([r, inv_var], axis=1, keys=["r", "inv_var"]).dropna()

rows = {}
for f in FACTORS:
    d = managed(f)
    c = d.r.std() / (d.inv_var * d.r).std()                              # full-sample constant: uses the future
    w = c * d.inv_var
    fit = sm.OLS(w * d.r, sm.add_constant(d.r)).fit()
    rows[f] = {"Sharpe, original": sharpe(d.r), "Sharpe, managed": sharpe(w * d.r), "alpha, % a year": fit.params.iloc[0] * 1200,
               "t(alpha)": fit.tvalues.iloc[0], "largest weight": w.max()}
print(f"{d.index[0]} to {d.index[-1]}; Moreira and Muir report a market alpha of 4.9% a year for 1926-2015")
pd.DataFrame(rows).T

Two warnings before you believe the table. The constant c uses the whole sample, so nobody could have known it in 1927. And the largest weight column shows the strategy sometimes holds the factor many times over, which few investors can do.

## Part 3. In real time (in class)

Cederburg, O'Doherty, Wang and Yan (2020) ask what an investor could have done without hindsight. Across 103 strategies, 77 managed versions have positive in-sample alphas, but real-time versions earn less than the original strategy in most cases (72 of 103 in their main test).

Here is a simple real-time version: estimate c from data up to last month only (starting after ten years), and cap the weight at 2 (at most twice the original position).

In [ ]:
CAP = 2.0          # Your turn: 1.5, 3, or 100 (no cap)
rt = {}
for f in FACTORS:
    d = managed(f)
    c_rt = (d.r.expanding(120).std() / (d.inv_var * d.r).expanding(120).std()).shift(1)
    w = (c_rt * d.inv_var).clip(upper=CAP)
    r_m = (w * d.r).dropna(); r_o = d.r.reindex(r_m.index)
    fit = sm.OLS(r_m, sm.add_constant(r_o)).fit()
    rt[f] = {"Sharpe, original": sharpe(r_o), "Sharpe, managed": sharpe(r_m), "alpha, % a year": fit.params.iloc[0] * 1200,
             "t(alpha)": fit.tvalues.iloc[0], "average |change in weight|": w.diff().abs().mean(),
             "Sharpe gain 1937-1981": sharpe(r_m.loc[:"1981-12"]) - sharpe(r_o.loc[:"1981-12"]),
             "Sharpe gain 1982-today": sharpe(r_m.loc["1982-01":]) - sharpe(r_o.loc["1982-01":])}
    if f == "Mom":
        crash = ("2009-03", "2009-05")
        print(f"Momentum, March-May 2009: original {(1 + r_o.loc[crash[0]:crash[1]]).prod() - 1:.0%}, "
              f"managed {(1 + r_m.loc[crash[0]:crash[1]]).prod() - 1:.0%} (weights {w.loc[crash[0]:crash[1]].round(2).tolist()})")
realtime = pd.DataFrame(rt).T
realtime

**Check yourself.**
1. Which factor's alpha survives in real time, and which one's mostly disappears?
2. Momentum crashed when markets rebounded in spring 2009 (Barroso and Santa-Clara, 2015). Why was the managed version barely invested at that point?
3. Set `CAP = 100`. What happens to the market's Sharpe ratio, and why might a real investor not be allowed to do that?

## Part 4. Break-even costs (in class)

Volatility management trades every month even when the factor's stocks don't change, because the *size* of the position changes. The break-even cost is the managed strategy's real-time alpha divided by how much its weight changes each month.

Two reference points for what trading actually costs:
- **Frazzini, Israel and Moskowitz** measure AQR's own trades, 1998&ndash;2013: an average market impact of about 11 basis points per dollar traded, an order of magnitude below earlier academic estimates.
- **Novy-Marx and Velikov** use effective bid-ask spreads estimated from daily prices, which are much higher for small and illiquid stocks.

Trading the market itself through futures or an ETF costs a basis point or two. Trading HML or momentum means trading hundreds of stocks on both sides.

In [ ]:
COST_BP = 11       # Your turn: cost per dollar traded, in basis points. Try 5, 25, 50
be = pd.DataFrame({
    "real-time alpha, % a year": realtime["alpha, % a year"],
    "dollars traded a month": realtime["average |change in weight|"],
    "break-even cost, bp": realtime["alpha, % a year"] / 1200 / realtime["average |change in weight|"] * 1e4,
})
be[f"alpha after {COST_BP} bp, % a year"] = be["real-time alpha, % a year"] - be["dollars traded a month"] * COST_BP / 1e4 * 1200
print("Volatility-managed factors:"); print(be.round(2))
print("\nPart 1's anomaly strategies (all stocks): break-even cost, bp"); print(turnover["break-even cost, bp"].round(0))

This understates the cost of managing HML and momentum: changing the position's size is only part of the trading. The factor itself turns over too, and you pay for that whether or not you manage its volatility. Barroso and Detzel (2021) add both and find that, after costs, volatility management generally adds nothing for factors other than the market.

**Check yourself.**
1. At 11 bp a dollar, which strategies in this notebook still make money? At 50 bp?
2. Why would the same strategy cost more for a $10 billion fund than for a $10 million one? (Frazzini, Israel and Moskowitz's "break-even fund size".)
3. Novy-Marx and Velikov find that a *buy/hold spread* is the most effective way to cut costs: buy a stock when it enters the top fifth, but sell only when it drops out of the top two-fifths. Why does that help more for momentum than for value?

## Part 5. Discuss (in class)

1. Moreira and Muir's in-sample alpha for the market is large; the real-time version is much smaller. Is this the same lesson as Week 3's &Sigma;<sup>&minus;1</sup>&mu;, or a different one?
2. A volatility-targeting fund sells when volatility jumps. What happens to prices when many funds do that at once? (Week 6 comes back to this with August 2007 and March 2020.)
3. Fill in the last row of Session 6's checklist, "worth it after turnover and costs?", for value, momentum and low volatility.
4. Which strategy from this course would you trust with a $10 billion fund, and which only with $10 million?